# Goto introduction

ReachySDK for Reachy 2 offers you methods to make movements with the arms and head, controlling the target position in several ways, choosing the duration of the movement, or even the interpolation mode. We called them **Goto**.

Those methods work the same way on the arms, on the head, and on the mobile base.

The methods to use in order to control the robot are:
-  for the arms:  
    - **`goto()`**: depending on the parameter entered, you can control either :
        - the joint value of each joint in degrees : *list of 7 values (joint space)*
        - the end-effector position in the robot frame of reference : *4x4 homogeneous matrix (cartesian space)*
    - **`translate_by()`** and **`rotate_by()`** : you can translate or rotate the position of the end-effector in space, in robot frame or gripper frame

- for the head:  
    - **`goto()`**: depending on the parameter entered, you can control either :
        - the joint value of each head joint in degrees : *list of  3 values (joint space)*
        - the head orientation in the robot frame : *quaternion (cartesian space)*
        > Be careful that, between the joint and cartesian spaces, there is a 10-degree difference in pitch : to have the head looking forward, in joint space you have to put rpy = [0,10,0] whereas in cartesian space, it's the equivalent of [0,0,0].
    
    - **`look_at()`**: you control the head by giving a point in the robot coordinate system the head will look at
    - **`rotate_by()`**: you can rotate the head in relation to its current position, by setting roll, pitch and yaw values in degrees, either in relation to the robot's frame of reference or to the head.

- for the mobile base:
    - **`goto()`**: you control the mobile base by giving a position to reach in the odometry frame of the mobile base


Grippers and antennas also support goto commands:

- for the grippers:
    - **`goto()`**: you control the position or opening of the gripper

- for the antennas:
    - **`goto()`**: you control the position of the antennas


> The following tutorial only shows examples for the arms, head and mobile_base, but most of the description is also true for the goto commands on the grippers and antennas.

## Initialize your robot

Connect to your robot:

In [5]:
from reachy2_sdk import ReachySDK
import time 

reachy = ReachySDK(host='192.168.10.172')  

Local API version (1.0.21) is different from the robot's API version (1.0.19).
Some features may not work properly.
Please update the reachy2_core image on the robot to ensure compatibility, or downgrade your local reachy2_sdk package.
Failed to connect to video server with error: Failed to connect to video service: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "recvmsg:Connection reset by peer"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_status:14, grpc_message:"recvmsg:Connection reset by peer"}"
>.
ReachySDK.video will not be available.
This Reachy is in REAL mode :
⚠️  Be careful, you're controlling the PHYSICAL Reachy.



Turn on Reachy so the motors can be controlled, and set in the default posture :

In [6]:
reachy.turn_on()

reachy.goto_posture()

GoToHomeId(head=id: 8
, r_arm=id: 9
, l_arm=id: 10
)

This Reachy is in REAL mode :
⚠️  Be careful, you're controlling the PHYSICAL Reachy.



`goto_posture()` is a convenient function to configure the posture of all Reachy parts at once. 

It accepts the parameters : 
- *default* (i.e. `goto_posture('default')`), which gives Reachy's default pose with arms outstretched on either side of the body,
- *elbow_90* (i.e. `goto_posture('elbow_90')`), in which Reachy has the two forearms parallel to the ground. 

These standard poses can be useful when you want to start a new task from a known position.

> This example is available in [set_default_posture.py](set_default_posture.py)

## Set your First Move

Let's move Reachy's right arm. The *goto* allows to control the 7 degrees of freedom of the arm at once (see [getting_started](1_getting_started.ipynb))

In [7]:
goto_1 = reachy.r_arm.goto([0, 10, -10, -90, 0, 0, 0])

print(f'goto 1 {goto_1}')

goto 1 id: 59



This method returns an id, that you can use to get information on this movement or to cancel this movement. Store this id in a variable (*goto_1* here) to be able to use it further. All the goto methods return an id.

The parameters to give are a little different between the parts that are controlled based on joints (arms and head) and the mobile base, but let's see as both work.

## Goto commands - arms and head

The parts that can be controlled using joints, i.e:
- reachy.l_arm
- reachy.r_arm
- reachy.head

used goto that are defined by 3 parameters : 
- the **joint commands**, as a list of articular degree values (7 for the arms and 3 for the head)
- the **duration**, in seconds - *set to 2 by default*
- the **interpolation mode**, 'linear' or 'minimum_jerk' - *set to 'minimum_jerk' by default*


#### Goto duration 

You can give a custom duration for the execution of the movements, as shown in the examples above : 

In [8]:
reachy.head.goto([20, 20, -10], duration = 3)
reachy.l_arm.goto([0, -10, 10, -90, 0, 0, 0], duration = 5)


# Doing:
reachy.l_arm.goto([0, -10, 0, 0, 0, 0, 0])
# will lead to the same result as:
reachy.l_arm.goto([0, -10, 0, 0, 0, 0, 0], duration = 2)

id: 63

> Default duration is **2 seconds**.

You **cannot set a duration to 0 second**. This will raise an exception in your code!

In [9]:
reachy.l_arm.goto([0, 0, 0, 0, 0, 0, 0], duration = 0) # raises an exception

ValueError: duration cannot be set to 0.

#### Goto interpolation mode

The goto methods generates a trajectory between the present position and the goal position. This trajectory is then interpolated at a predefined frequency (100Hz) to compute all intermediary target positions that should be followed before reaching the final goal position. Depending on the interpolation mode chosen, you can have a better control over speed and acceleration.

Two interpolation modes are available when sending a goto command:
- the **linear** interpolation mode
- the **minimum-jerk** interpolation mode

Both trajectories start and finish at the same point but don't follow the same intermediate positions. The minimum jerk will slowly accelerate at the begining and slowly decelerate at the end. This makes the movements more natural.

You can specify the interpolation mode by setting the **`interpolation_mode`** argument when calling the method:

In [10]:
reachy.goto_posture('default', wait = True)
reachy.head.goto([20, 20, -10], interpolation_mode='linear')
reachy.l_arm.goto([0, -10, 10, -90, 0, 0, 0], interpolation_mode='linear')

id: 70

> Default interpolation mode is **minimum_jerk**.

In [11]:
# Doing:
reachy.l_arm.goto([0, -10, 0, 0, 0, 0, 0])
# will lead to the same result as:
reachy.l_arm.goto([0, -10, 0, 0, 0, 0, 0], interpolation_mode='minimum_jerk')


id: 72

## Goto commands - mobile base

> *Pass this section if you do not have a robot with a mobile base or if you are using the robot in fake mode*

The goto parameters for the mobile base are a little different, as you cannot ask the mobile base to reach a target position in a given time. Instead of trying to reach the given position in a given time, the robot will take as long as needed to reach its goal, or be interrupted by the timeout you can set as parameter.

The parameters are:
- the **x** target, in meters
- the **y** target, in meters
- the **theta** target, in degrees by default
- the **distance_tolerance** and **angle_tolerance**, to define how close to your target the robot must be to consider the target has been reached
- a **timeout** value, to stop the movement if the target has been reached after a certain amount of time

In [29]:
# Rotation to reach a 90-degrees-rotation around the odom coordinate system
reachy.mobile_base.goto(x=0, y=0, theta=-100)

id: 122

#### Goto tolerances

You can modify the tolerances around the goal position you want to reach. Tuning the tolerances will modify the precision of the position at arrival, but also modify the time the mobile base will take before considering its movement over. You will learn more about these tolerances in the [mobile_base tutorial](5_mobile_base.ipynb).

#### Goto timeout

You don't have any duration parameters to control the mobile base, but you can use the timeout to stop the mobile base goto after a certain amount of time.  
The movement is stopped if the target has not been reached after the duration of this timeout. In the case the target has been reached before the timeout is over, the timeout will have no impact.

In [13]:
# Set back the robot in position 0
reachy.mobile_base.goto(x=0, y=0, theta=0)

# Move forward by 30cm
reachy.mobile_base.goto(x=0.3, y=0, theta=0)

id: 75

In [14]:
reachy.mobile_base.goto(x=0.6, y=0, theta=0)

id: 76

If we print the current position of the robot, we can it has reached the goal position:

In [15]:
print(f"x: {round(reachy.mobile_base.odometry['x'], 1)}")
print(f"y: {round(reachy.mobile_base.odometry['y'], 1)}")
print(f"theta: {round(reachy.mobile_base.odometry['theta'], 1)}")

x: 0.6
y: -0.0
theta: 1.6


Let's do the same movement by adding a timetout to the forward goto:

In [16]:
# Set back the robot in position 0
reachy.mobile_base.goto(x=0, y=0, theta=0)

# Move forward by 30cm, with a timeout of 0.5 seconds
reachy.mobile_base.goto(x=0.3, y=0, theta=0, timeout=0.8)

id: 78

If we print the current position, we can see the robot was stopped before reaching the target:

In [17]:
print(f"x: {round(reachy.mobile_base.odometry['x'], 1)}")
print(f"y: {round(reachy.mobile_base.odometry['y'], 1)}")
print(f"theta: {round(reachy.mobile_base.odometry['theta'], 1)}")

x: 0.1
y: -0.0
theta: 1.3


> Default timeout is **100 seconds**.

## Goto execution

There are two important concepts to be aware of : 
- gotos are stacked for a part (i.e. they run one after another),
- but each part is independent (i.e. a goto for the left arm will run in parallel with a goto for the right arm).

All the following section is applicable to all parts (arms, head and mobile base).

### Goto is non-blocking for other parts 

It means you can send a goto command on different parts, it won't wait for the movement to be executed on the first part to execute the other one, but will follow the timing of your code.

Let's take an example with a motion sequence : 

- Start by returning the robot to its neutral position.

In [18]:
reachy.goto_posture('default')

GoToHomeId(head=id: 81
, r_arm=id: 82
, l_arm=id: 83
)

- Send a goto on both arms, with a delay between them. 

In [19]:
reachy.l_arm.goto([0, 0, 10, -90, 0, 0, 15], duration = 3)
time.sleep(1)
reachy.r_arm.goto([0, 0, -10, -90, 0, 0, -15], duration = 2)

id: 85

This sequence will take 3 seconds to execute, as the right arm will start its movement 1 second after the left arm has started its own movement. They will finish at the same time.

### Goto is blocking and stacked for a part

It means that you can send several goto commands on a part one after another without any delay, they will be played in this order, but will wait for the previous goto to be finished.  

Let's take an example with the following sequence:

In [20]:
reachy.goto_posture('default')
reachy.l_arm.goto([0, 0, 15, -90, 0, 0, 15], duration = 3)
reachy.l_arm.goto([0, -10, 0, 0, 0, 0, 0], duration = 2)
reachy.l_arm.goto([0, 0, 15, -90, 0, 0, 15], duration = 3)

id: 93

This sequence will take 8 seconds to execute, as each movement on the left arm will wait for the previous before starting.  

Nevertheless, you can still send goto commands to other parts.

In [ ]:
reachy.goto_posture('default')

reachy.l_arm.goto([0, 0, 15, -90, 0, 0, 15], duration = 3)      #1
time.sleep(1)
reachy.l_arm.goto([0, -10, 0, 0, 0, 0, 0], duration = 2)        #2
reachy.l_arm.goto([0, 0, 15, -90, 0, 0, 15], duration = 3)      #3
reachy.r_arm.goto([0, 0, -15, -90, 0, 0, -15], duration = 2)    #4

id: 102

This sequence will still take 8 seconds to execute:
- commands #1, #2 and #3 are sent to the left arm. They will be stacked on the left arm, and the `time.sleep(1)` won't have any effect . When received, command #2 will simply wait 2 seconds rather than 3 secondes in the previous example.
- commands #4 is sent on the right arm, where no movement is processed. It will then start 1 second after command #1 has started, and will then be over approximatively at the same time.

The sequence execution order is #1, #4, #2, #3.

So how can a left arm goto wait for a right arm move? That's simple using the parameter *wait* in goto functions ! 

### Wait parameter

As you could see earlier in the goto_posture() command, we can set the parameter *wait = True* in goto functions for the execution of the program to wait for the end of the movement before going on. 


In [22]:
reachy.goto_posture('default', wait = True)
print('Default posture : done')
r_goto_1 = reachy.r_arm.goto([0, 5, -15, -90, 0, 0, -10], duration = 2, wait = True)
print('Right move : done')
r_goto_2 = reachy.l_arm.goto([0, -5, 15, -90, 0, 0, 10], duration = 2, wait = True)
print('Left move : done')
reachy.goto_posture('default', wait = True)
print('Default posture : done')

Default posture : done
Right move : done
Left move : done
Default posture : done


### Goto state

For a specific goto, you may want to know its current state. You can get information on whether the goto is finished or not using:

- **`is_goto_finished()`**: return True if the movement is over, but also if it won't be played because it has been cancelled for example

You can also ask for the request sent for a specific goto with:

- **`get_goto_request()`**: return the part concerned by the command, as well as the goal position and other parameters of the request


Let's take an example:

In [25]:
goto_1 = reachy.l_arm.goto([0, 0, 0, -60, 0, 0, 0], duration = 3)
if reachy.mobile_base is not None:
    goto_2 = reachy.mobile_base.goto(0.2, 0, 0)

time.sleep(1)

# Goto is currently being played
goto1_is_finished = reachy.is_goto_finished(goto_1)
print(f'After 1 second, goto 1 is finished : {goto1_is_finished}\n')

time.sleep(3)

# Goto is now over
goto1_is_finished = reachy.is_goto_finished(goto_1)
print(f'After 4 seconds, goto 1 is finished : {goto1_is_finished}')

After 1 second, goto 1 is finished : False

After 4 seconds, goto 1 is finished : True


We then have for the l_arm the goto_1

In [26]:
print(f"Part: {reachy.get_goto_request(goto_1).part}")
print(f"Request: {reachy.get_goto_request(goto_1).request}")

Part: l_arm
Request: JointsRequest(target=TargetJointsRequest(joints=[0.0, 0.0, 0.0, -60.000001669652114, 0.0, 0.0, 0.0], pose=None), duration=3.0, mode='minimum_jerk', interpolation_space='joint_space', elliptical_parameters=None)


And the mobile base the second one:

In [27]:
if reachy.mobile_base is not None:
    print(f"Part: {reachy.get_goto_request(goto_2).part}")
    print(f"Request: {reachy.get_goto_request(goto_2).request}")
else: 
    print("Cell content ignored, no mobile_base is available")

Part: mobile_base
Request: OdometryRequest(target={'x': 0.20000000298023224, 'y': 0.0, 'theta': 0.0}, timeout=100.0, distance_tolerance=0.05000000074505806, angle_tolerance=4.99999985454646)


You get information on the part involved, the target joint values, the duration of the movement, and the interpolation mode. 

### Part execution state

As the sequence can become complex, you can get information for each part on its current status, to know which movement is being played and know which others are waiting to be played.  
For each part, the following methods are available:
- **`get_goto_playing()`**: will return the id of the currently played goto on the part
- **`get_goto_queue()`**: will return the ids of all stacked goto commands waiting to be played on the part

Those methods are called at the part level, to get info on the state of the part.  

Let's take an example. 

In [28]:
# Write a sequence for the left arm
goto_1 = reachy.l_arm.goto([0, -15, 15, -90, 0, 0, 0], duration = 3)
goto_2 = reachy.l_arm.goto([0, -10, 0, 0, 0, 0, 0], duration = 2)
goto_3 = reachy.l_arm.goto([0, -15, 15, -90, 0, 0, 0], duration = 3)

print(f'goto 1: {goto_1.id}, goto 2: {goto_2.id}, goto 3: {goto_3.id}')

# Goto #1 is currently playing
current_goto = reachy.l_arm.get_goto_playing()
print(f'current goto : {current_goto.id}')
print(f'l_arm queue length: {len(reachy.l_arm.get_goto_queue())} gotos waiting to be played.')

goto 1: 119, goto 2: 120, goto 3: 121
current goto : 119
l_arm queue length: 2 gotos waiting to be played.


## Goto cancellation

If you want to modify the queue of goto commands on a part, or interrupt the movement being played, you can cancel goto commands at any time.  

Cancellations work the same way for all parts (arms, head and mobile base)

### Single goto cancellation

To cancel a single movement, currently playing or stacked in a part's queue, use its id and call `cancel_goto_by_id()` from reachy. It will stop the robot at its current position.

In [31]:
reachy.goto_posture('default', wait = True)
goto_1 = reachy.l_arm.goto([0, 15, 15, -90, 10, 0, 0], duration = 3)
goto_2 = reachy.head.goto([30, 0, 0], duration = 3)

time.sleep(1)
reachy.cancel_goto_by_id(goto_1)

ack: true

### Multiple gotos cancellation

To cancel all gotos at once, you can call the `cancel_all_goto()` methods.  
This method can be called at the level you want to act, which can be either **reachy** or a **specific part**. 

#### All gotos

For example, if you want to cancel all gotos on all parts:

In [32]:
reachy.goto_posture('default', wait = True)

# Send a sequence of gotos
reachy.head.goto([20, 30, -10], duration = 3)
reachy.l_arm.goto([0, 0, 0, -90, 0, 0, 0], duration = 3)
reachy.l_arm.goto([0, 0, 0, 0, 0, 0, 0], duration = 3)

time.sleep(1.5)

# Cancel all gotos
reachy.cancel_all_goto()

print(f"Length of l_arm goto queue : {len(reachy.l_arm.get_goto_queue())}")

Length of l_arm goto queue : 0


All movements are cancelled, even the movement stacked in the left arm queue which will never be played.  

#### All gotos for one part

If you only want to cancel movement on the left arm:

In [33]:
reachy.goto_posture('default', wait = True)

# Send a sequence of gotos
reachy.head.goto([20, 30, -10], duration=3)
reachy.l_arm.goto([0, 0, 0, -90, 0, 0, 0], duration = 3)
reachy.l_arm.goto([0, 0, 0, 0, 0, 0, 0], duration = 2)

time.sleep(1)

# Cancel gotos on left arm only
reachy.l_arm.cancel_all_goto()

ack: true

The movement on the head will continue, but all the movements of the left will be stopped and the left arm queue cleaned.

In [34]:
reachy.goto_posture('default', wait=True)
reachy.turn_off()

True